# Brazos aprendidos servidos en vivo desde un documento portable: la misma predicción en Python y en el navegador

**Nivel 4: Más allá del estado del arte** · Requiere: 05-learned-arms/01-fitting-a-learned-arm-on-nine-sites, 06-design-response/02-a-burden-and-spacing-surface · Tiempo de lectura: unos 30 minutos

## Objetivos

La aplicación web recalcula cada modelo, los aprendidos incluidos, sobre un diseño que el lector cambia, sin servidor Python. Eso funciona porque cada brazo aprendido se exporta una vez a un documento JSON portable que un navegador puede evaluar (afirmación BS-5). Este cuaderno abre uno de esos documentos, lo evalúa con el lector de referencia del motor, y muestra qué lo hace confiable.

Al terminar habrá:

- abierto el archivo de modelos del ámbito Enusa y listado sus siete brazos, sus tipos y su tamaño en disco;
- comprobado su dirección de contenido, y luego mostrado que alterar un solo número la cambia;
- evaluado los fixtures que lleva, comparado la predicción portable con un ajuste nuevo del motor sobre las mismas filas de entrenamiento, y movido una palanca de diseño para ver que ambos la siguen.

## Preparación

Instale los paquetes con los pines exactos de `notebooks/requirements.txt`. El motor es `blastfrag`, publicado en PyPI; los otros paquetes son su base numérica y la biblioteca de gráficos.

```bash
pip install blastfrag==0.4.0 numpy==2.5.4 matplotlib==3.11.2
```

Esta celda aún no importa nada del cuaderno: lee las versiones instaladas desde los metadatos de los paquetes y las imprime, de modo que la salida registra qué motor produjo cada número que sigue.

In [1]:
# Imprima la versión instalada de cada paquete que usa este cuaderno
from importlib.metadata import version

for name in ("blastfrag", "numpy", "matplotlib",):
    print(f"{name}=={version(name)}")

blastfrag==0.4.0
numpy==2.5.4
matplotlib==3.11.2


## Datos

| Fuente | Titular y URL | Clase de licencia | Archivo que lee el cuaderno |
|---|---|---|---|
| Las filas de entrenamiento | Hudaverdi, Kulatilake y Kuzu 2010, https://doi.org/10.1002/nag.957 | hechos numéricos reutilizados con cita; el artículo nunca se distribuye | el corpus dentro del paquete `blastfrag` |
| El archivo de modelos del ámbito Enusa: los siete brazos ajustados sin ningún tiro de Enusa, con sus fixtures | este repositorio (MIT), escrito por la ejecución offline | la licencia propia del producto | `data/derived/models/Enusa.json` |
| El índice, que lista cada archivo de modelos con su resumen | este repositorio (MIT), escrito por la ejecución offline | la licencia propia del producto | `data/derived/manifests/index.json` |

El lector del formato portable es `bf.predict_portable`, en el motor. El navegador tiene su propio lector en TypeScript; la prueba del producto `frontend/test/parity.test.ts` lo contrasta con los documentos versionados.

## Método

Un brazo aprendido ajustado no es un pickle ni un servidor de modelos. El motor escribe cada uno como un documento JSON en un esquema versionado (`blastfrag.portable/v1`) que guarda exactamente lo que necesita una predicción: los árboles como arreglos de hijos izquierdo y derecho, variables de corte y umbrales para el bosque y el modelo potenciado; las constantes de estandarización; los vectores de soporte y coeficientes duales para el regresor de vectores de soporte; los pesos de capa para la red; los coeficientes para la ley de potencia reajustada; y, para el conjunto de apilamiento, una referencia al bosque y al modelo potenciado más el metaaprendiz. Leerlo necesita solo aritmética, así que un lector escrito en TypeScript puede reproducirlo en un navegador.

Un archivo de modelos agrupa los siete brazos de un ámbito de entrenamiento, aquí los nueve sitios que quedan al dejar Enusa fuera, y cierra con una dirección de contenido, el SHA-256 de la serialización canónica del archivo. También lleva fixtures: vectores de entrada y, por brazo, la predicción a precisión completa del modelo ajustado original, contra la cual puede probarse cualquier lector del formato.

## Estado del arte

- Los predictores aprendidos publicados para este corpus se informan mediante sus puntajes y sus parámetros: los anchos de la red (Kulatilake et al. 2012, https://doi.org/10.1007/s10706-012-9496-3), los núcleos y constantes de vectores de soporte (Amoako et al. 2022, https://doi.org/10.3390/mining2020013; Sui et al. 2025, https://doi.org/10.3390/app15031254) y la tasa de aprendizaje de boosting (Sui et al. 2025).
- Las fuentes que tuvo la investigación no registran una forma ejecutable de ninguno que un lector pueda aplicar a un diseño nuevo. Quien quiere la predicción para un diseño debe reentrenar el modelo desde los parámetros.
- La manera convencional de servir un modelo aprendido es un proceso Python detrás de una API, que el despliegue estático del producto no tiene.

## Más allá del estado del arte

### Paso 1: abrir el archivo de modelos y comprobar su dirección de contenido

La celda carga el archivo de modelos de Enusa y el índice, lista los brazos con su tipo, y recalcula la dirección de contenido con la propia función `digest` del pipeline, que hace el resumen de la serialización canónica del contenido sin su campo `digest`. Compara el resultado con el resumen guardado en el archivo y con el que lista el índice.

In [2]:
# Importaciones, el archivo de modelos del ámbito Enusa, sus brazos y su dirección de contenido
import copy
import json
import sys
from pathlib import Path

import blastfrag as bf
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path("../../../data-pipeline")))
from pipeline.cases.fragmenta_cases import Variant
from pipeline.model.blasts import apply_variant
from pipeline.stages.export import digest

DERIVED = Path("../../../data/derived")
path = DERIVED / "models" / "Enusa.json"
document = json.loads(path.read_text(encoding="utf-8"))
index = json.loads((DERIVED / "manifests" / "index.json").read_text(encoding="utf-8"))
print("ámbito:", document["scope"], "| sitio excluido:", document["held_out_site"], "| filas de entrenamiento:", document["n_training_rows"], "| schema:", document["schema"], "/", document["portable_schema"])
print("tamaño del archivo:", path.stat().st_size, "bytes")
for name, arm in document["arms"].items():
    print(f"  {name:<22}{arm['kind']:<20}{len(json.dumps(arm)):>9} caracteres de JSON")

payload = {k: v for k, v in document.items() if k != "digest"}
recomputed = digest(payload)
listed = next(m["digest"] for m in index["models"] if m["scope"] == "Enusa")
print("resumen recalculado igual al guardado:", recomputed == document["digest"], "| igual al índice:", recomputed == listed)

ámbito: Enusa | sitio excluido: Enusa | filas de entrenamiento: 85 | schema: fragmenta.models/v1 / blastfrag.portable/v1
tamaño del archivo: 238371 bytes
  kuznetsov-transfer    kuznetsov-transfer        891 caracteres de JSON
  published-neural-net  network                 24782 caracteres de JSON
  random-forest         forest                 168687 caracteres de JSON
  refitted-regression   power-law                 992 caracteres de JSON
  stacking              stacking                  890 caracteres de JSON
  svr-rbf               svr                      9114 caracteres de JSON
  xgboost               xgboost                 40032 caracteres de JSON
resumen recalculado igual al guardado: True | igual al índice: True


La celda siguiente altera una copia: cambia un valor de hoja del bosque aleatorio en una centésima de milímetro y recalcula la dirección. Una dirección de contenido que no se moviera sería inútil.

In [3]:
# Cambiar un número de una copia y recalcular la dirección de contenido
tampered = copy.deepcopy(payload)
tampered["arms"]["random-forest"]["trees"][0]["t"][-1] += 1e-5
print("la dirección de la copia alterada difiere:", digest(tampered) != document["digest"])

la dirección de la copia alterada difiere: True


### Paso 2: los fixtures y un ajuste nuevo

Un lector del formato es correcto si reproduce los fixtures. La celda evalúa cada entrada de fixture con `predict_portable` y compara con los valores esperados guardados a precisión completa. El documento de apilamiento guarda referencias al bosque y al modelo potenciado, así que la celda las resuelve primero. Para el brazo de transferencia el valor es el factor de roca, no un tamaño.

Luego va un paso más: ajusta de nuevo los brazos del motor sobre las mismas filas de entrenamiento (todo tiro fuera de Enusa) y compara la predicción portable con el brazo nuevo en los 97 tiros.

In [4]:
# Portable contra fixtures guardados, y portable contra un ajuste nuevo del motor sobre las mismas filas
arms = document["arms"]

def readable(name):
    # Un documento de apilamiento remite al bosque y al modelo potenciado, así que se los incluye antes de leer
    doc = copy.deepcopy(arms[name])
    if doc["kind"] == "stacking":
        doc["forest"] = arms[doc["forest"]["ref"]]
        doc["boosting"] = arms[doc["boosting"]["ref"]]
    return doc

fixtures = document["fixtures"]
print("entradas de fixture:", len(fixtures["inputs"]))
print(f"{'brazo':<22}{'mayor diferencia con los fixtures':>44}")
for name in arms:
    gaps = []
    for item, want in zip(fixtures["inputs"], fixtures["expected"][name]):
        value, _ = bf.predict_portable(readable(name), item["features"])
        if value is not None and want is not None:
            gaps.append(abs(value - want))
    print(f"{name:<22}{max(gaps):>44.2e}")

train = bf.load_training_corpus()
fit_rows = [b for b in train if b.site != document["held_out_site"]]
print("ajuste nuevo con", len(fit_rows), "filas, comparado en los", len(train), "tiros")
factories = bf.default_arms()
for name in [n for n in arms if n != "kuznetsov-transfer"]:
    fresh = factories[name]().fit(fit_rows)
    gaps = []
    for blast in train:
        engine = fresh.predict_one(blast).x50_m
        portable, _ = bf.predict_portable(readable(name), blast)
        if engine is not None and portable is not None:
            gaps.append(abs(engine - portable))
    print(f"{name:<22} mayor diferencia motor contra portable = {max(gaps):.2e} m  ({len(gaps)} tiros)")

entradas de fixture: 116
brazo                            mayor diferencia con los fixtures
kuznetsov-transfer                                        0.00e+00
published-neural-net                                      8.88e-16
random-forest                                             0.00e+00
refitted-regression                                       0.00e+00
stacking                                                  0.00e+00
svr-rbf                                                   2.78e-15
xgboost                                                   0.00e+00
ajuste nuevo con 85 filas, comparado en los 97 tiros
published-neural-net   mayor diferencia motor contra portable = 8.88e-16 m  (97 tiros)
random-forest          mayor diferencia motor contra portable = 0.00e+00 m  (97 tiros)
refitted-regression    mayor diferencia motor contra portable = 0.00e+00 m  (97 tiros)
stacking               mayor diferencia motor contra portable = 0.00e+00 m  (97 tiros)
svr-rbf                mayor diferenci

### Paso 3: mover una palanca de diseño

El sentido de un documento portable es que la predicción siga un diseño que el lector cambia. La celda aplica la variante `burden-tight` del producto (bordo 15 por ciento menor) a un tiro de Enusa y predice el diseño base y el perturbado con el documento portable y con el brazo nuevo del motor, para los brazos de gradient boosting y de apilamiento.

In [5]:
# El diseño base y uno con bordo más cerrado, predichos desde el documento portable y desde el motor
blast = next(b for b in train if b.blast_id == "En1")
tight = apply_variant(blast, Variant("burden-tight", "", "", "B_over_D", 0.85))
print(f"{'brazo':<12}{'diseño':<22}{'portable (m)':>16}{'motor (m)':>14}")
for name in ("xgboost", "stacking"):
    fresh = factories[name]().fit(fit_rows)
    for label, design in (("base", blast), ("burden-tight", tight)):
        portable, _ = bf.predict_portable(readable(name), design)
        print(f"{name:<12}{label:<22}{portable:>16.4f}{fresh.predict_one(design).x50_m:>14.4f}")

brazo       diseño                    portable (m)     motor (m)
xgboost     base                            0.4601        0.4601
xgboost     burden-tight                    0.4600        0.4600
stacking    base                            0.4606        0.4606
stacking    burden-tight                    0.4604        0.4604


## Resultados

**El documento es un modelo completo y autocontenido.** Los siete brazos del ámbito Enusa ocupan unos cientos de kilobytes de JSON y necesitan solo aritmética. El lector de referencia reproduce cada fixture a precisión de flotante.

**Es el mismo modelo que el del motor.** Un ajuste nuevo del motor sobre las mismas 85 filas de entrenamiento da las mismas predicciones que el documento portable en los 97 tiros, a redondeo de punto flotante (diferencias del orden de 1e-15 m). El navegador recalcula por tanto el modelo que se ajustó y puntuó, no una reimplementación que podría derivar.

**Es evidente ante alteraciones.** La dirección de contenido recalculada con la función del pipeline iguala la guardada y la del índice, y el cambio de un número en una hoja la mueve. Un documento obsoleto o editado no puede pasar por el versionado.

**Sigue un diseño.** Cerrar el bordo mueve juntas la predicción portable y la del motor; para los brazos basados en árboles el movimiento es pequeño y puede ser cero, como mostró la escalera del cuaderno anterior.

**Qué no muestra esto.** La paridad de dos lectores dice que el navegador calcula la misma función, no que la función sea exacta: los brazos aprendidos aún puntúan cerca de cero o bajo cero con el sitio fuera. Y el documento del brazo de transferencia devuelve un factor de roca, no un tamaño, así que el lector debe aplicarle la ecuación clásica.

## Resumen

- Cada brazo aprendido se exporta una vez a un documento JSON portable; el navegador lo evalúa solo con aritmética (afirmación BS-5).
- El lector de referencia reproduce los fixtures guardados, y un ajuste nuevo del motor sobre las mismas filas da las mismas predicciones.
- Una dirección de contenido hace al documento evidente ante alteraciones, y la predicción sigue una palanca de diseño.
- Esto cierra el conjunto: los datos, la ecuación clásica, las curvas, el protocolo, los brazos aprendidos y la respuesta en vivo.

## Referencias

- Kulatilake, Hudaverdi y Wu 2012, https://doi.org/10.1007/s10706-012-9496-3 (la red)
- Amoako, Jha y Zhong 2022, https://doi.org/10.3390/mining2020013 (regresión de vectores de soporte)
- Sui, Zhou, Zhao, Yang y Zou 2025, https://doi.org/10.3390/app15031254 (bosque, boosting, apilamiento)
- Hudaverdi, Kulatilake y Kuzu 2010, https://doi.org/10.1002/nag.957 (el corpus)
- `blastfrag` 0.4.0, el motor, en PyPI: https://pypi.org/project/blastfrag/